# AEGIS — zokastech.fr — Apache 2.0 / MIT

# Démonstration interactive AEGIS

Les chaînes utilisées dans les cellules sont des **exemples factices** (ne pas y coller de vraies PII). Les réponses viennent du **moteur Rust** derrière la passerelle (`/v1/analyze`, `/v1/anonymize`) — pas d’API simulée dans ce notebook.

**Prérequis** : passerelle joignable.

- **`docker compose up`** (fichier racine) : la gateway est en **HTTPS** sur **`https://127.0.0.1:8443`** (certificat auto-signé dans l’image). Le défaut du notebook correspond à ça (`verify=False` pour le TLS local).
- **`docker compose -f docker-compose.dev.yml`** : HTTP **`http://127.0.0.1:8080`** — définir `AEGIS_BASE_URL` en conséquence.

Variables utiles : `AEGIS_BASE_URL`, `AEGIS_TLS_VERIFY=1` pour exiger une vérification TLS stricte (HTTPS avec certificat valide).

**Erreur `Connection reset by peer` ?** Souvent : **HTTP** vers un serveur **HTTPS**, ou mauvais **port** (stack `docker compose` racine : **https://127.0.0.1:8443**, pas `http://…:8080` sur l’hôte). Réexécutez la cellule suivante : elle **détecte** automatiquement la bonne base.

**Erreur `401 Unauthorized` ?** La passerelle exige une clé API (`X-API-Key`) sauf si le compose dev définit `AEGIS_SECURITY_DEVELOPMENT_DISABLE_AUTH=true`. Sinon : `export AEGIS_API_KEY=votre_clé` avant le noyau, ou utilisez `docker compose -f docker-compose.dev.yml` à jour.


In [87]:
# Décommentez si nécessaire dans votre environnement Jupyter :
# %pip install -q requests pandas

import json
import os

import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def _req_kw_for_base(url: str) -> dict:
    kw: dict = {}
    if url.startswith("https://"):
        verify = os.environ.get("AEGIS_TLS_VERIFY", "0").lower() in ("1", "true", "yes")
        kw["verify"] = verify
    api_key = os.environ.get("AEGIS_API_KEY", "").strip()
    if api_key:
        header = os.environ.get("AEGIS_API_KEY_HEADER", "X-API-Key").strip() or "X-API-Key"
        kw["headers"] = {header: api_key}
    return kw


def _probe(url: str, **req_kw) -> bool:
    u = url.rstrip("/")
    probe_kw = {k: v for k, v in req_kw.items() if k != "headers"}
    for path in ("/livez", "/health/live"):
        try:
            r = requests.get(f"{u}{path}", timeout=4, **probe_kw)
            if r.status_code in (200, 204):
                return True
        except requests.RequestException:
            continue
    return False


def _discover() -> tuple[str, dict]:
    forced = os.environ.get("AEGIS_BASE_URL", "").strip()
    if forced:
        b = forced.rstrip("/")
        return b, _req_kw_for_base(b)

    candidates = [
        ("https://127.0.0.1:8443", {"verify": False}),
        ("https://localhost:8443", {"verify": False}),
        ("http://127.0.0.1:8080", {}),
        ("http://localhost:8080", {}),
    ]
    for base, kw in candidates:
        if _probe(base, **kw):
            print(f"Passerelle détectée : {base}")
            merged = {**kw, **_req_kw_for_base(base)}
            return base.rstrip("/"), merged

    print(
        "Aucune sonde OK sur 8443 (HTTPS) ni 8080 (HTTP). "
        "Défaut : http://127.0.0.1:8080 — vérifiez `docker compose ps`."
    )
    b = "http://127.0.0.1:8080"
    return b, _req_kw_for_base(b)


BASE, REQ_KW = _discover()
print("Requêtes avec :", BASE, "|", REQ_KW if REQ_KW else "HTTP (sans TLS client)")
if os.environ.get("AEGIS_API_KEY", "").strip():
    print("En-tête clé API :", os.environ.get("AEGIS_API_KEY_HEADER", "X-API-Key"))
BASE, REQ_KW


Aucune sonde OK sur 8443 (HTTPS) ni 8080 (HTTP). Défaut : http://127.0.0.1:8080 — vérifiez `docker compose ps`.
Requêtes avec : http://127.0.0.1:8080 | HTTP (sans TLS client)


('http://127.0.0.1:8080', {})

In [89]:
# Texte d’exemple factif — le moteur réel est invoqué côté gateway.
TEXT = "Synthetic row: jane@customer.zokastech.com booked +33 6 55 44 33 22."
ANALYSIS_CFG = json.dumps(
    {"language": "fr", "pipeline_level": 2, "score_threshold": 0.5}
)
_url = f"{BASE.rstrip('/')}/v1/analyze"
r = requests.post(
    _url,
    json={"text": TEXT, "analysis_config_json": ANALYSIS_CFG},
    timeout=60,
    **REQ_KW,
)
try:
    r.raise_for_status()
except requests.HTTPError as e:
    body = (r.text or "")[:500]
    raise RuntimeError(f"{e} — corps: {body!r}") from e
print(json.dumps(r.json(), indent=2)[:2000])

{
  "result": {
    "entities": [
      {
        "end": 42,
        "entity_type": "EMAIL",
        "metadata": {},
        "recognizer_name": "mock",
        "score": 0.99,
        "start": 19,
        "text": "@customer.zokastech.com"
      }
    ],
    "processing_time_ms": 1,
    "text_length": 68
  }
}


In [91]:
cfg = {
    "operators_by_entity": {
        "EMAIL": {"operator_type": "mask", "params": {"keep_last": "4", "mask_char": "*"}},
        "PHONE": {"operator_type": "redact", "params": {"replacement": "[PHONE]"}},
    }
}
r2 = requests.post(
    f"{BASE}/v1/anonymize",
    json={"text": TEXT, "config_json": json.dumps(cfg)},
    timeout=60,
    **REQ_KW,
)
r2.raise_for_status()
print(json.dumps(r2.json(), indent=2)[:2000])

{
  "result": {
    "anonymized": {
      "spans": [],
      "text": "[REDACTED]",
      "tokens": []
    },
    "analysis": {
      "entities": [
        {
          "end": 42,
          "entity_type": "EMAIL",
          "metadata": {},
          "recognizer_name": "mock",
          "score": 0.99,
          "start": 19,
          "text": "@customer.zokastech.com"
        }
      ],
      "processing_time_ms": 1,
      "text_length": 68
    }
  }
}


In [83]:
import pandas as pd

df = pd.DataFrame(
    {
        "id": range(20),
        "note": [f"user_{i}@demo.example invalid phone +33 6 {i:02d} 00 00 00 00" for i in range(20)],
    }
)
df.head()

,id,note
0,0,user_0@demo.example invalid phone +33 6 00 00 ...
1,1,user_1@demo.example invalid phone +33 6 01 00 ...
2,2,user_2@demo.example invalid phone +33 6 02 00 ...
3,3,user_3@demo.example invalid phone +33 6 03 00 ...
4,4,user_4@demo.example invalid phone +33 6 04 00 ...


## Suite

- Scripts CLI : `examples/python/quickstart.py`, `pandas_pipeline.py`, …
- Documentation MkDocs : `mkdocs serve` à la racine du dépôt.
